In [1]:
from google.cloud import bigquery
import pandas as pd
import requests

In [2]:
# Weather Archive API data frame: 2025 year sample
# Date: 2025-01-01 to 2025-12-31
# Location: London; latitude: 51.5; longitude: -0.1

url = 'https://archive-api.open-meteo.com/v1/archive'
params = {
	"latitude": 51.5,
	"longitude": -0.1,
	"start_date": "2025-01-01",
	"end_date": "2025-12-31",
	"hourly": [
        "temperature_2m",
        "wind_speed_10m",
        "wind_speed_100m",
        "wind_direction_10m",
        "wind_direction_100m",
        "wind_gusts_10m",
        "cloud_cover",
        "cloud_cover_low",
        "cloud_cover_mid",
        "cloud_cover_high",
        "shortwave_radiation",
        "direct_radiation",
        "diffuse_radiation",
        "direct_normal_irradiance",
        "global_tilted_irradiance",
        "terrestrial_radiation",
        "pressure_msl",
        "snowfall"
        ],
}

weather_a_data = requests.get(url, params=params).json()
weather_a_df = pd.DataFrame(weather_a_data["hourly"])

# coverting to datetime and rest index
weather_a_df['time'] = pd.to_datetime(weather_a_df['time'])
weather_a_df = weather_a_df.set_index('time')


print(weather_a_df.head())

                     temperature_2m  wind_speed_10m  wind_speed_100m  \
time                                                                   
2025-01-01 00:00:00            10.5            25.1             48.0   
2025-01-01 01:00:00            11.1            25.1             46.3   
2025-01-01 02:00:00            11.1            22.9             43.6   
2025-01-01 03:00:00            11.1            22.0             42.1   
2025-01-01 04:00:00            10.9            23.4             43.6   

                     wind_direction_10m  wind_direction_100m  wind_gusts_10m  \
time                                                                           
2025-01-01 00:00:00                 225                  228            55.1   
2025-01-01 01:00:00                 224                  225            58.3   
2025-01-01 02:00:00                 220                  222            60.5   
2025-01-01 03:00:00                 224                  225            61.9   
2025-01-01 04:0

In [3]:
weather_a_df.shape

(8760, 18)

In [4]:
# Upload to BQ on GCP
PROJECT = 'gridzero-489711'
DATASET ='historical_weather'
TABLE = '02_full_params_weather_2025'

table = f'{PROJECT}.{DATASET}.{TABLE}'

client = bigquery.Client(project=PROJECT)

write_mode = 'WRITE_TRUNCATE'
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(weather_a_df, table, job_config=job_config)
result = job.result()

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [5]:
# convert timestamp to half-hourly (hhr) for joining
weather_a_df_hhr = weather_a_df.resample('30min').ffill()

print(weather_a_df_hhr.head())
print(weather_a_df_hhr.shape)

                     temperature_2m  wind_speed_10m  wind_speed_100m  \
time                                                                   
2025-01-01 00:00:00            10.5            25.1             48.0   
2025-01-01 00:30:00            10.5            25.1             48.0   
2025-01-01 01:00:00            11.1            25.1             46.3   
2025-01-01 01:30:00            11.1            25.1             46.3   
2025-01-01 02:00:00            11.1            22.9             43.6   

                     wind_direction_10m  wind_direction_100m  wind_gusts_10m  \
time                                                                           
2025-01-01 00:00:00                 225                  228            55.1   
2025-01-01 00:30:00                 225                  228            55.1   
2025-01-01 01:00:00                 224                  225            58.3   
2025-01-01 01:30:00                 224                  225            58.3   
2025-01-01 02:0

In [6]:
# Upload to BQ on GCP
PROJECT = 'gridzero-489711'
DATASET ='historical_weather'
TABLE = '03_halfhourly_full_params_weather_2025'

table = f'{PROJECT}.{DATASET}.{TABLE}'

client = bigquery.Client(project=PROJECT)

write_mode = 'WRITE_TRUNCATE'
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(weather_a_df_hhr, table, job_config=job_config)
result = job.result()

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [7]:
# Selected params only
cols = weather_a_df.columns.tolist()
print(cols)

cols_select = [
    'temperature_2m',
    'wind_speed_100m',
    'wind_gusts_10m',
    'cloud_cover',
    'shortwave_radiation',
    'direct_radiation',
    'diffuse_radiation',
    'pressure_msl',
    'snowfall'
]

# cols_to_drop = [
#     'wind_direction_10m',
#     'wind_direction_100m',
#     'cloud_cover_low',
#     'cloud_cover_mid',
#     'cloud_cover_high',
#     'direct_normal_irradiance',
#     'global_tilted_irradiance',
#     'terrestrial_radiation',
# ]
# weather_a_df_select = weather_a_df.drop(columns=cols_to_drop)

['temperature_2m', 'wind_speed_10m', 'wind_speed_100m', 'wind_direction_10m', 'wind_direction_100m', 'wind_gusts_10m', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'global_tilted_irradiance', 'terrestrial_radiation', 'pressure_msl', 'snowfall']


In [8]:
# selected hourly df
weather_a_df_select = weather_a_df[cols_select]
print(weather_a_df_select.head)

<bound method NDFrame.head of                      temperature_2m  wind_speed_100m  wind_gusts_10m  \
time                                                                   
2025-01-01 00:00:00            10.5             48.0            55.1   
2025-01-01 01:00:00            11.1             46.3            58.3   
2025-01-01 02:00:00            11.1             43.6            60.5   
2025-01-01 03:00:00            11.1             42.1            61.9   
2025-01-01 04:00:00            10.9             43.6            58.3   
...                             ...              ...             ...   
2025-12-31 19:00:00             0.2             22.2            22.0   
2025-12-31 20:00:00            -0.3             22.0            22.3   
2025-12-31 21:00:00            -0.8             22.5            23.4   
2025-12-31 22:00:00            -0.9             23.3            23.8   
2025-12-31 23:00:00            -0.8             23.4            23.8   

                     cloud_cover 

In [9]:
# Upload to BQ on GCP
PROJECT = 'gridzero-489711'
DATASET ='historical_weather'
TABLE = '04_select_params_weather_2025'

table = f'{PROJECT}.{DATASET}.{TABLE}'

client = bigquery.Client(project=PROJECT)

write_mode = 'WRITE_TRUNCATE'
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(weather_a_df_select, table, job_config=job_config)
result = job.result()

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [10]:
# selected half hourly df
weather_a_df_hhr_select = weather_a_df_hhr[cols_select]
print(weather_a_df_hhr_select.head)

<bound method NDFrame.head of                      temperature_2m  wind_speed_100m  wind_gusts_10m  \
time                                                                   
2025-01-01 00:00:00            10.5             48.0            55.1   
2025-01-01 00:30:00            10.5             48.0            55.1   
2025-01-01 01:00:00            11.1             46.3            58.3   
2025-01-01 01:30:00            11.1             46.3            58.3   
2025-01-01 02:00:00            11.1             43.6            60.5   
...                             ...              ...             ...   
2025-12-31 21:00:00            -0.8             22.5            23.4   
2025-12-31 21:30:00            -0.8             22.5            23.4   
2025-12-31 22:00:00            -0.9             23.3            23.8   
2025-12-31 22:30:00            -0.9             23.3            23.8   
2025-12-31 23:00:00            -0.8             23.4            23.8   

                     cloud_cover 

In [11]:
# Upload to BQ on GCP
PROJECT = 'gridzero-489711'
DATASET ='historical_weather'
TABLE = '05_halfhourly_select_params_weather_2025'

table = f'{PROJECT}.{DATASET}.{TABLE}'

client = bigquery.Client(project=PROJECT)

write_mode = 'WRITE_TRUNCATE'
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(weather_a_df_hhr_select, table, job_config=job_config)
result = job.result()

/Users/madeleine/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
